In [1]:
import sys
import os
import logging
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ==============================================================================
# CÉLULA 1: SETUP E CONFIGURAÇÕES INICIAIS
# ==============================================================================

# 1. Configuração do Diretório (Garante conexão com o Drive se rodar no Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    caminho_projeto = '/content/drive/MyDrive/1) PESQUISA/ESALQ Data Science/tcc/tema_classificacao_queda_arvore/git/tcc_risco_queda_v-pub'
    sys.path.append(caminho_projeto)
    os.chdir(caminho_projeto)
    print("✅ Diretório Colab configurado:", os.getcwd())
except ImportError:
    # Se estiver rodando localmente (não no Colab)
    print("✅ Rodando em ambiente local:", os.getcwd())

Mounted at /content/drive
✅ Diretório Colab configurado: /content/drive/MyDrive/1) PESQUISA/ESALQ Data Science/tcc/tema_classificacao_queda_arvore/git/tcc_risco_queda_v-pub


# COMEÇA AQUI

In [81]:
import pandas as pd
import geopandas as gpd
import numpy as np

pd.set_option('display.max_columns', None)

# ATENÇÃO: Substitua pelo caminho real da pasta principal do seu projeto no Drive
caminho_projeto = '/content/drive/MyDrive/1) PESQUISA/ESALQ Data Science/tcc/tema_classificacao_queda_arvore/git/tcc_risco_queda_v-pub'

if caminho_projeto not in sys.path:
    sys.path.append(caminho_projeto)

# 4. Importa as suas funções da pasta src
from src.spatial_features import (
    preparar_indice_espacial,
    calcular_idag,
    identificar_aiv,
    calcular_dva,
    calcular_icc,
    get_azimute
)

from src.ETL_functions import imputar_por_proximidade

In [100]:
# ==========================================
# 2. CARREGAMENTO DOS DATASETS
# ==========================================
arvores = gpd.read_file('data/processed/arvores.gpkg').to_crs(epsg=31983)
vias = gpd.read_file('data/processed/vias.gpkg').to_crs(epsg=31983)
vias_calcadas = gpd.read_file('data/processed/vias_calcadas_DEDUP.gpkg').to_crs(epsg=31983)
quedas = gpd.read_file('data/processed/quedas_DEDUP_2014_2017.gpkg').to_crs(epsg=31983)

# Renomear as colunas de vias_calcadas para o padrão do Dicionário de Dados
vias_calcadas = vias_calcadas.rename(columns={
    'cvc_codlog': 'segmento_id',
    'cc_lg_min_min': 'via_calcada_largura_min',
    'cc_dec_max_max': 'via_calcada_declinacao_max'
    # Se houver coluna de comprimento, ex: 'extensao_km': 'via_extensao_km'
})

In [101]:
# TRATAMENTO DE NANS E IMPUTAÇÃO

#Imputação Agrupada (Mediana pelo Tipo de Logradouro: RUA, AVENIDA, PRAÇA, etc.)

# Calculando medianas agrupadas e preenchendo os NaNs
vias_calcadas['via_calcada_largura_min'] = vias_calcadas.groupby('cvc_tplogr')['via_calcada_largura_min'].transform(
    lambda x: x.fillna(x.median())
)
vias_calcadas['via_calcada_declinacao_max'] = vias_calcadas.groupby('cvc_tplogr')['via_calcada_declinacao_max'].transform(
    lambda x: x.fillna(x.median())
)

# 2.3 Fallback de Segurança
# Caso exista alguma categoria viária onde TUDO seja nulo, usamos a mediana global da cidade
mediana_global_largura = vias_calcadas['via_calcada_largura_min'].median()
mediana_global_inclinacao = vias_calcadas['via_calcada_declinacao_max'].median()

vias_calcadas['via_calcada_largura_min'] = vias_calcadas['via_calcada_largura_min'].fillna(mediana_global_largura)
vias_calcadas['via_calcada_declinacao_max'] = vias_calcadas['via_calcada_declinacao_max'].fillna(mediana_global_inclinacao)

# 2.4 Criação da feature de extensão viária (usada depois no ICC)
vias_calcadas['via_extensao_km'] = (vias_calcadas.geometry.length / 1000).astype('float32')

In [102]:
# ==========================================
# 0. PREPARAÇÃO: O NOVO "ID GEOMÉTRICO"
# ==========================================
# Resetamos o index para garantir uma ordem limpa e criamos o spatial_uid
vias_calcadas = vias_calcadas.reset_index(drop=True)
vias_calcadas['spatial_uid'] = vias_calcadas.index

# ==========================================
# 1. ESCALA INDIVIDUAL: MÉTRICAS DAS ÁRVORES
# ==========================================
print("1. Calculando métricas individuais das árvores...")
coords, tree = preparar_indice_espacial(arvores)

arvores['idag_15_indiv'] = calcular_idag(coords, tree, raio_corte=15.0, beta=2.0)
arvores['is_isolada'] = identificar_aiv(coords, tree, limite_isolamento=15.0)
arvores['dva_3_indiv'] = calcular_dva(coords, tree, vizinhos=3)
arvores['idag_15_log_indiv'] = np.log1p(arvores['idag_15_indiv']).astype('float32')

# ==========================================
# 2. SJOIN: ÁRVORES -> VIAS (Garantia 1:1)
# ==========================================
print("2. Associando árvores às vias geometricamente...")
# O how='inner' remove árvores que estão no meio do nada (longe de vias)
arvores_com_via = gpd.sjoin_nearest(
    arvores,
    vias_calcadas[['spatial_uid', 'geometry']],
    how='inner',
    max_distance=20.0
)

# A TRAVA DO 1:1 -> Se uma árvore caiu a distâncias iguais de duas vias,
# o index dela ficará duplicado. Nós dropamos mantendo apenas a primeira via.
arvores_com_via = arvores_com_via[~arvores_com_via.index.duplicated(keep='first')]

# ==========================================
# 3. AGREGAÇÃO: ESCALA DA VIA (Versão Blindada)
# ==========================================
print("3. Agregando métricas arbóreas por via...")

# 3.1 Faz a agregação
df_agregado = arvores_com_via.groupby('spatial_uid').agg(
    via_arvores_contagem=('spatial_uid', 'count'),
    stat_aiv_arvores_isoladas=('is_isolada', 'sum'),
    stat_dva_vizinhanca_dist_med=('dva_3_indiv', 'mean'),
    stat_dva_vizinhanca_dist_std=('dva_3_indiv', 'std'),
    stat_idag_densidade_grav_med=('idag_15_log_indiv', 'mean'),
    stat_idag_densidade_grav_std=('idag_15_log_indiv', 'std')
).reset_index()

# 3.2 Preenchendo nulos de vias com apenas 1 árvore (onde o std dá NaN)
# A checagem 'in' evita que o código quebre se o df_agregado estiver vazio
if 'stat_dva_vizinhanca_dist_std' in df_agregado.columns:
    df_agregado['stat_dva_vizinhanca_dist_std'] = df_agregado['stat_dva_vizinhanca_dist_std'].fillna(0)
if 'stat_idag_densidade_grav_std' in df_agregado.columns:
    df_agregado['stat_idag_densidade_grav_std'] = df_agregado['stat_idag_densidade_grav_std'].fillna(0)

# 3.3 O PASSO CRÍTICO: Acoplando as agregações de volta na base principal
vias_calcadas = vias_calcadas.merge(df_agregado, on='spatial_uid', how='left')

# Lista de colunas alvo
cols_fill = [
    'via_arvores_contagem', 'stat_aiv_arvores_isoladas', 'stat_dva_vizinhanca_dist_med',
    'stat_dva_vizinhanca_dist_std', 'stat_idag_densidade_grav_med', 'stat_idag_densidade_grav_std'
]

# 3.4 Trava de segurança: Se por algum motivo as colunas não vieram no merge, nós as criamos zeradas
for col in cols_fill:
    if col not in vias_calcadas.columns:
        vias_calcadas[col] = 0.0

# Agora o fillna é 100% seguro
vias_calcadas[cols_fill] = vias_calcadas[cols_fill].fillna(0)

print("Agregação concluída com sucesso! ✅")

# ==========================================
# 4. INFRAESTRUTURA E CÁLCULOS GEOMÉTRICOS (ICC e Azimute)
# ==========================================
print("4. Calculando ICC e Azimute...")
vias_calcadas['via_extensao_km'] = (vias_calcadas.geometry.length / 1000).astype('float32')

# Calculando ICC (garanta que a coluna de largura não tenha NaNs neste momento)
vias_calcadas['via_icc_confinamento_idx'] = calcular_icc(
    df=vias_calcadas,
    col_arvores='via_arvores_contagem',
    col_largura='via_calcada_largura_min',
    col_comprimento_km='via_extensao_km'
).astype('float32')

# Azimute (usando a função otimizada para polígonos)
vias_calcadas['via_azimute_graus'] = vias_calcadas.geometry.apply(get_azimute).astype('float32')

# ==========================================
# 5. TARGET: QUEDAS -> VIAS (Garantia 1:1)
# ==========================================
print("5. Associando ocorrências de queda (Target)...")
quedas_com_via = gpd.sjoin_nearest(
    quedas,
    vias_calcadas[['spatial_uid', 'geometry']],
    how='inner',
    max_distance=50.0 # Tolerância maior para GPS de bombeiros/defesa civil
)

# A TRAVA DO 1:1 -> Uma mesma queda não pode contar para duas vias diferentes
quedas_com_via = quedas_com_via[~quedas_com_via.index.duplicated(keep='first')]

# Contando quantas quedas caíram em cada spatial_uid
agg_quedas = quedas_com_via.groupby('spatial_uid').size().reset_index(name='target_historico_quedas')

# Acoplando o target na base final
df_matriz_final = vias_calcadas.merge(agg_quedas, on='spatial_uid', how='left')
df_matriz_final['target_historico_quedas'] = df_matriz_final['target_historico_quedas'].fillna(0).astype('int64')
df_matriz_final['target_queda_bool'] = (df_matriz_final['target_historico_quedas'] > 0).astype('int64')

# Limpando o identificador temporário e o index
df_matriz_final = df_matriz_final.drop(columns=['spatial_uid'])

print(f"Pipeline concluído! Matriz final com {len(df_matriz_final)} linhas preservadas. ✅")

1. Calculando métricas individuais das árvores...
2. Associando árvores às vias geometricamente...
3. Agregando métricas arbóreas por via...
Agregação concluída com sucesso! ✅
4. Calculando ICC e Azimute...
5. Associando ocorrências de queda (Target)...
Pipeline concluído! Matriz final com 220064 linhas preservadas. ✅


In [119]:
ordem_colunas = [
    # 1. Identificadores
    'cvc_nomelg',
    'cvc_tplogr',
    'cvc_classe',

    # 2. Infraestrutura
    'via_extensao_km',
    'via_azimute_graus',
    'via_calcada_largura_min',
    'via_calcada_declinacao_max',

    # 3. Árvores e Conflito
    'via_arvores_contagem',
    'via_icc_confinamento_idx',

    # 4. Métricas Espaciais
    'stat_aiv_arvores_isoladas',
    'stat_dva_vizinhanca_dist_med',
    'stat_dva_vizinhanca_dist_std',
    'stat_idag_densidade_grav_med',
    'stat_idag_densidade_grav_std',

    # 5. Geometria
    'geometry',

    # 6. Targets
    'target_historico_quedas',
    'target_queda_bool'
]

# Aplicando a reordenação no GeoDataFrame
df_matriz_final = df_matriz_final[ordem_colunas]

In [120]:
# Selecionando apenas vias com árvores
df = df_matriz_final.copy()
df_fil = df[df['via_arvores_contagem'] > 0].reset_index(drop=True)

In [154]:
# PERFIL DO DATASET

# ARVORES
df_fil['stat_aiv_arvores_isoladas'].sum() # 79597 isoladas
df_fil['via_arvores_contagem'].sum() # 651778 arvores

# QUEDAS
df_fil['target_historico_quedas'].sum() # 9530 quedas (2014-2017)

# VIAS
# 110957 segmentos de vias com árvores
len(df_fil[df_fil['via_arvores_contagem'] >= 1])
# 7326 segmentos de vias com ao menos 1 queda (2014-2017)
df_fil['target_queda_bool'].sum()

np.int64(9530)

In [123]:
# Export
df_fil.to_file('data/processed/dataset_vias_features_espaciais_DEDUP.gpkg', driver='gpkg')

In [158]:
df_fil.info()
df_fil.head()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 110957 entries, 0 to 110956
Data columns (total 18 columns):
 #   Column                        Non-Null Count   Dtype   
---  ------                        --------------   -----   
 0   cvc_nomelg                    110494 non-null  object  
 1   cvc_tplogr                    109399 non-null  object  
 2   cvc_classe                    110957 non-null  object  
 3   via_extensao_km               110957 non-null  float32 
 4   via_azimute_graus             110957 non-null  float32 
 5   via_calcada_largura_min       110957 non-null  float64 
 6   via_calcada_declinacao_max    110957 non-null  float64 
 7   via_arvores_contagem          110957 non-null  float64 
 8   via_icc_confinamento_idx      110957 non-null  float32 
 9   stat_aiv_arvores_isoladas     110957 non-null  float64 
 10  stat_dva_vizinhanca_dist_med  110957 non-null  float64 
 11  stat_dva_vizinhanca_dist_std  110957 non-null  float64 
 12  stat_idag_densidade_gr

,cvc_nomelg,cvc_tplogr,cvc_classe,via_extensao_km,via_azimute_graus,via_calcada_largura_min,via_calcada_declinacao_max,via_arvores_contagem,via_icc_confinamento_idx,stat_aiv_arvores_isoladas,stat_dva_vizinhanca_dist_med,stat_dva_vizinhanca_dist_std,stat_idag_densidade_grav_med,stat_idag_densidade_grav_std,geometry,target_historico_quedas,target_queda_bool,geom_wkb
0,R RAFAEL DA SILVA E SOUSA,R,LOCAL,0.179425,172.858521,1.79,5.76,1.0,0.003096,1.0,27.767830,0.000000,0.000000,0.000000,"MULTIPOLYGON (((349356.509 7393959.048, 349379...",0,0,b'\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...
1,R GAMA BARROS,R,LOCAL,0.223504,11.401561,1.31,4.99,1.0,0.003390,1.0,34.411237,0.000000,0.000000,0.000000,"MULTIPOLYGON (((340793.317 7390162.461, 340813...",0,0,b'\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...
2,R EMANUEL ARAUJO,R,LOCAL,0.280840,103.194031,0.88,9.94,2.0,0.008002,2.0,40.695976,8.326930,0.000000,0.000000,"MULTIPOLYGON (((357705.361 7397897.498, 357705...",0,0,"b""\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03..."
3,R DO HORTO,R,COLETORA,0.580185,37.750301,0.80,8.81,5.0,0.010639,5.0,37.334874,7.979385,0.000000,0.000000,"MULTIPOLYGON (((333621.195 7404440.741, 333625...",0,0,b'\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...
4,R VITAL RIFARTO,R,LOCAL,0.171458,151.630066,0.73,13.61,5.0,0.039408,0.0,10.939023,3.258623,0.042079,0.020966,"MULTIPOLYGON (((317362.616 7382432.234, 317389...",0,0,b'\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...


# Juntando com socio, infra e geologica (00_dataset)

In [124]:
# merge features espaciais, socio, infra e geologicas
master = pd.read_parquet('data/processed/00_dataset_queda_MASTER.parquet')

In [128]:
import geopandas as gpd

print("1. Deduplicando a base master...")
# Como estamos olhando para dentro do próprio arquivo, o WKB funciona bem para dedup
master['geom_wkb'] = master.geometry.to_wkb()
master_dedup = master.drop_duplicates(subset=['geom_wkb'], keep='first').copy()
master_dedup = master_dedup.drop(columns=['geom_wkb'])

print(f"Linhas na master antes: {len(master)} | Após dedup: {len(master_dedup)}")

colunas_puxar = [
    'socio_zona_fiscal_cat', 'socio_vulnerabilidade_idx', 'socio_densidade_demog_hab_ha',
    'infra_escolas_contagem', 'infra_semaforos_contagem', 'infra_iluminacao_contagem',
    'infra_onibus_pontos_contagem', 'infra_uso_solo_cat',
    'geo_declividade_terreno_cat', 'geo_relevo_tipo_cat', 'geo_litologia_tipo_cat', 'geo_solo_mole_area_m2'
]

print("2. Cruzando as bases com tolerância de precisão (0.1 metros)...")
# Garantir o CRS idêntico
df_fil = df_fil.to_crs(epsg=31983)
master_dedup = master_dedup.to_crs(epsg=31983)

# Removemos as colunas vazias que falharam no teste anterior (se existirem)
df_fil_clean = df_fil.drop(columns=colunas_puxar, errors='ignore')

# Usamos sjoin_nearest com max_distance de 10cm.
# Isso conecta polígonos que são visualmente idênticos, mas têm variações de float.
df_fil_merged = gpd.sjoin_nearest(
    df_fil_clean,
    master_dedup[colunas_puxar + ['geometry']],
    how='left',
    max_distance=0.1,
    distance_col='dist_micrometrica'
)

# 3. Trava de segurança 1:1
# Remove duplicatas caso o sjoin_nearest encontre mais de um vizinho no raio de 10cm
df_fil_merged = df_fil_merged[~df_fil_merged.index.duplicated(keep='first')]

print(f"Merge concluído! Linhas preservadas: {len(df_fil_merged)}")
print(f"Sucesso! Registros de Uso do Solo encontrados: {df_fil_merged['infra_uso_solo_cat'].notnull().sum()}")

1. Deduplicando a base master...
Linhas na master antes: 108226 | Após dedup: 107718
2. Cruzando as bases com tolerância de precisão (0.1 metros)...
Merge concluído! Linhas preservadas: 110957
Sucesso! Registros de Uso do Solo encontrados: 109889


In [132]:
# Imputando por proximidade

# Lista de colunas que seu .info() mostrou que têm nulos
colunas_vazias = [
    'socio_zona_fiscal_cat',
    'socio_vulnerabilidade_idx',
    'socio_densidade_demog_hab_ha',
    'geo_relevo_tipo_cat',
    'geo_litologia_tipo_cat'
]

# Executa a função
gdf_limpo = imputar_por_proximidade(df_fil_merged, colunas_vazias)

Imputando 1068 valores na coluna: socio_zona_fiscal_cat...
Imputando 1068 valores na coluna: socio_vulnerabilidade_idx...
Imputando 1068 valores na coluna: socio_densidade_demog_hab_ha...
Imputando 14578 valores na coluna: geo_relevo_tipo_cat...
Imputando 14578 valores na coluna: geo_litologia_tipo_cat...


In [134]:
# Lista de colunas organizada por grupos temáticos
colunas_ordenadas = [
    # 1. IDENTIFICAÇÃO (Quem e onde é o segmento)
    'cvc_nomelg',
    'cvc_tplogr',
    'cvc_classe',

    # 2. CARACTERÍSTICAS DA VIA (Geometria viária e calçada)
    'via_extensao_km',
    'via_azimute_graus',
    'via_calcada_largura_min',
    'via_calcada_declinacao_max',

    # 3. VEGETAÇÃO E ESTATÍSTICAS ESPACIAIS (O objeto de estudo principal)
    'via_arvores_contagem',
    'via_icc_confinamento_idx',
    'stat_aiv_arvores_isoladas',
    'stat_dva_vizinhanca_dist_med',
    'stat_dva_vizinhanca_dist_std',
    'stat_idag_densidade_grav_med',
    'stat_idag_densidade_grav_std',

    # 4. CONTEXTO SOCIOECONÔMICO (Fatores humanos)
    'socio_zona_fiscal_cat',
    'socio_vulnerabilidade_idx',
    'socio_densidade_demog_hab_ha',

    # 5. INFRAESTRUTURA URBANA (Fatores de entorno)
    'infra_uso_solo_cat',
    'infra_escolas_contagem',
    'infra_semaforos_contagem',
    'infra_iluminacao_contagem',
    'infra_onibus_pontos_contagem',

    # 6. GEOGRAFIA E GEOLOGIA (Fatores de solo e relevo)
    'geo_declividade_terreno_cat',
    'geo_relevo_tipo_cat',
    'geo_litologia_tipo_cat',
    'geo_solo_mole_area_m2',

    # 7. TARGETS (O que o modelo deve prever)
    'target_historico_quedas',
    'target_queda_bool',

    # 8. GEOMETRIA (Sempre por último por convenção do GeoPandas)
    'geometry'
]

# Aplicando a nova ordem ao GeoDataFrame
gdf_limpo = gdf_limpo[colunas_ordenadas]

In [153]:
# PERFIL DO DATASET

# ARVORES
gdf_limpo['stat_aiv_arvores_isoladas'].sum() # 79597 isoladas
gdf_limpo['via_arvores_contagem'].sum() # 651778 arvores

# QUEDAS
gdf_limpo['target_historico_quedas'].sum() # 9530 quedas (2014-2017)

# VIAS
# 110957 segmentos de vias com árvores
len(gdf_limpo[gdf_limpo['via_arvores_contagem'] >= 1])
# 7326 segmentos de vias com ao menos 1 queda (2014-2017)
gdf_limpo['target_queda_bool'].sum()

np.int64(9530)

In [155]:
gdf_limpo.to_file('data/processed/dataset_all_features_imputed_DEDUP.gpkg', driver='gpkg')

In [161]:
gdf_limpo.info()
gdf_limpo.head()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 110957 entries, 0 to 110956
Data columns (total 29 columns):
 #   Column                        Non-Null Count   Dtype   
---  ------                        --------------   -----   
 0   cvc_nomelg                    110494 non-null  object  
 1   cvc_tplogr                    109399 non-null  object  
 2   cvc_classe                    110957 non-null  object  
 3   via_extensao_km               110957 non-null  float32 
 4   via_azimute_graus             110957 non-null  float32 
 5   via_calcada_largura_min       110957 non-null  float64 
 6   via_calcada_declinacao_max    110957 non-null  float64 
 7   via_arvores_contagem          110957 non-null  float64 
 8   via_icc_confinamento_idx      110957 non-null  float32 
 9   stat_aiv_arvores_isoladas     110957 non-null  float64 
 10  stat_dva_vizinhanca_dist_med  110957 non-null  float64 
 11  stat_dva_vizinhanca_dist_std  110957 non-null  float64 
 12  stat_idag_densidade_grav_me

,cvc_nomelg,cvc_tplogr,cvc_classe,via_extensao_km,via_azimute_graus,via_calcada_largura_min,via_calcada_declinacao_max,via_arvores_contagem,via_icc_confinamento_idx,stat_aiv_arvores_isoladas,stat_dva_vizinhanca_dist_med,stat_dva_vizinhanca_dist_std,stat_idag_densidade_grav_med,stat_idag_densidade_grav_std,socio_zona_fiscal_cat,socio_vulnerabilidade_idx,socio_densidade_demog_hab_ha,infra_uso_solo_cat,infra_escolas_contagem,infra_semaforos_contagem,infra_iluminacao_contagem,infra_onibus_pontos_contagem,geo_declividade_terreno_cat,geo_relevo_tipo_cat,geo_litologia_tipo_cat,geo_solo_mole_area_m2,target_historico_quedas,target_queda_bool,geometry
0,R RAFAEL DA SILVA E SOUSA,R,LOCAL,0.179425,172.858521,1.79,5.76,1.0,0.003096,1.0,27.767830,0.000000,0.000000,0.000000,ZF-3,2,100.263542,COMRCIO E SERVIO HORIZONTAL,0.0,1.0,12.0,0.0,0,MORROTES,GNAISSES E MIGMATITOS,0.0,0,0,"MULTIPOLYGON (((349356.509 7393959.048, 349379..."
1,R GAMA BARROS,R,LOCAL,0.223504,11.401561,1.31,4.99,1.0,0.003390,1.0,34.411237,0.000000,0.000000,0.000000,ZF-2,2,70.215355,COMRCIO E SERVIO HORIZONTAL,0.0,0.0,8.0,0.0,1,COLINAS,LAMITOS ARENOSOS E ARENITOS,0.0,0,0,"MULTIPOLYGON (((340793.317 7390162.461, 340813..."
2,R EMANUEL ARAUJO,R,LOCAL,0.280840,103.194031,0.88,9.94,2.0,0.008002,2.0,40.695976,8.326930,0.000000,0.000000,ZF-3,5,240.415848,RESIDENCIAL HORIZONTAL BAIXO PADRO,0.0,0.0,3.0,0.0,2,MORROTES,LAMITOS ARENOSOS E ARENITOS,0.0,0,0,"MULTIPOLYGON (((357705.361 7397897.498, 357705..."
3,R DO HORTO,R,COLETORA,0.580185,37.750301,0.80,8.81,5.0,0.010639,5.0,37.334874,7.979385,0.000000,0.000000,ZF-2,2,70.117020,RESIDENCIAL HORIZONTAL MDIO PADRO,0.0,0.0,7.0,2.0,0,MORROTES,GRANITOS E GRANITÓIDES,0.0,0,0,"MULTIPOLYGON (((333621.195 7404440.741, 333625..."
4,R VITAL RIFARTO,R,LOCAL,0.171458,151.630066,0.73,13.61,5.0,0.039408,0.0,10.939023,3.258623,0.042079,0.020966,ZF-3,3,243.600357,COMRCIO E SERVIO HORIZONTAL,0.0,0.0,4.0,0.0,2,MORROTES,GNAISSES E MIGMATITOS,0.0,0,0,"MULTIPOLYGON (((317362.616 7382432.234, 317389..."
